# 12 · 混合精度训练：fp32 vs fp16 vs bf16

> **学习目标**：把同一个模型分别用 fp32 / fp16（带 GradScaler）/ bf16（autocast）训一遍，亲眼看精度、显存、时间的差异。理解 `GradScaler` 解决什么问题、`bf16` 为什么是现代默认。
>
> **预备**：notebook 06 已过（MNIST + 训练循环）。本机有 GPU + CUDA 才能完整看到收益。
>
> **为什么重要**：所有 7B+ 模型都用混合精度训。不会算「我的显存够不够」就上手训练，必然炸。同时会演示 grad clip + warmup —— 现代训练的标配。

In [ ]:
import torch, torch.nn as nn, torch.nn.functional as F
import time
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('torch:', torch.__version__, '| device:', device)
if device == 'cuda':
    print('GPU :', torch.cuda.get_device_name(0))
    print('bf16 支持:', torch.cuda.is_bf16_supported())
else:
    print('⚠ CPU 模式下 fp16 显存收益看不出来。bf16 / autocast 在 CPU 上也能跑，但速度/显存对比意义不大。')

## 0. 三种精度的本质区别

| 类型 | 总位数 | 指数位 | 尾数位 | 动态范围 | 精度 | 备注 |
|------|--------|--------|--------|----------|------|------|
| fp32 | 32 | 8  | 23 | ±10³⁸  | 高 | 训练 baseline |
| fp16 | 16 | 5  | 10 | ±6.5×10⁴ | 中 | 容易 underflow，**需要 GradScaler** |
| bf16 | 16 | 8  | 7  | ±10³⁸  | 低 | 范围 = fp32，精度差但**梯度安全**，**不需要 GradScaler** |

**一句话**：fp16 精度好但范围窄、要 scaler；bf16 范围与 fp32 同、精度差但训练不炸 —— **现代默认**。

In [ ]:
# 实测三种精度的最小正数和最大数
for dtype in [torch.float32, torch.float16, torch.bfloat16]:
    info = torch.finfo(dtype)
    print(f'{str(dtype):>14}  最小正常数 {info.tiny:.2e}   最大数 {info.max:.2e}   精度 ≈ {info.eps:.2e}')

print('\n关键观察：')
print('- fp16 最大约 6.5e4 —— 训 transformer 梯度容易溢出/下溢')
print('- bf16 最大与 fp32 相当 —— 不会溢出，但精度比 fp16 差')

## 1. 准备数据与模型（深一点的 MLP，让差异更明显）

用 `784 → 1024 → 1024 → 1024 → 10` 的 MLP，~3M 参数 —— 比 notebook 06 大 ~20 倍，显存/时间差异看得更清楚。

In [ ]:
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.1307,), (0.3081,))])
train_set = datasets.MNIST('./_mnist_data', train=True,  download=True, transform=transform)
test_set  = datasets.MNIST('./_mnist_data', train=False, download=True, transform=transform)
train_loader = DataLoader(train_set, batch_size=512, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_set,  batch_size=512, shuffle=False, num_workers=0)

class DeepMLP(nn.Module):
    def __init__(self, h=1024):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(784, h), nn.ReLU(),
            nn.Linear(h, h),   nn.ReLU(),
            nn.Linear(h, h),   nn.ReLU(),
            nn.Linear(h, 10),
        )
    def forward(self, x):
        return self.net(x)

n_params = sum(p.numel() for p in DeepMLP().parameters())
print(f'参数量: {n_params:,} ({n_params/1e6:.2f} M)')

## 2. 三个训练函数 —— 对照组

**关键差别**：
- fp32：baseline，啥都不改
- fp16 + autocast + **GradScaler**：scaler 把 loss 放大 → 反传 → 把梯度 unscale 回去（防 fp16 下溢）
- bf16 + autocast：直接套 autocast，**不需要 scaler**（bf16 范围与 fp32 同）

In [ ]:
EPOCHS = 2

def make_model_opt():
    torch.manual_seed(0)
    model = DeepMLP().to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3)
    return model, opt

@torch.no_grad()
def eval_acc(model):
    model.eval()
    correct = total = 0
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        correct += (model(x).argmax(1) == y).sum().item()
        total += y.size(0)
    model.train()
    return correct / total

def train_fp32():
    model, opt = make_model_opt()
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
    losses = []
    t0 = time.perf_counter()
    for epoch in range(EPOCHS):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            loss = F.cross_entropy(model(x), y)
            loss.backward()
            # 梯度裁剪：常见做法，防梯度爆炸
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            losses.append(loss.item())
    dt = time.perf_counter() - t0
    mem = torch.cuda.max_memory_allocated()/1024/1024 if device == 'cuda' else 0
    return dict(name='fp32', losses=losses, time=dt, mem_mb=mem, acc=eval_acc(model))

def train_fp16():
    if device != 'cuda':
        return dict(name='fp16', losses=[], time=0, mem_mb=0, acc=0, skip='only on GPU')
    model, opt = make_model_opt()
    torch.cuda.reset_peak_memory_stats()
    scaler = torch.amp.GradScaler('cuda')
    losses = []
    t0 = time.perf_counter()
    for epoch in range(EPOCHS):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.float16):
                loss = F.cross_entropy(model(x), y)
            scaler.scale(loss).backward()                    # scale up loss
            scaler.unscale_(opt)                              # 必须先 unscale 才能 clip
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            scaler.step(opt)
            scaler.update()
            losses.append(loss.item())
    dt = time.perf_counter() - t0
    mem = torch.cuda.max_memory_allocated()/1024/1024
    return dict(name='fp16 (GradScaler)', losses=losses, time=dt, mem_mb=mem, acc=eval_acc(model))

def train_bf16():
    if device != 'cuda' or not torch.cuda.is_bf16_supported():
        return dict(name='bf16', losses=[], time=0, mem_mb=0, acc=0, skip='requires GPU with bf16')
    model, opt = make_model_opt()
    torch.cuda.reset_peak_memory_stats()
    losses = []
    t0 = time.perf_counter()
    for epoch in range(EPOCHS):
        for x, y in train_loader:
            x, y = x.to(device), y.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast('cuda', dtype=torch.bfloat16):
                loss = F.cross_entropy(model(x), y)
            loss.backward()                                    # ⚠ 不用 scaler
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            opt.step()
            losses.append(loss.item())
    dt = time.perf_counter() - t0
    mem = torch.cuda.max_memory_allocated()/1024/1024
    return dict(name='bf16 (autocast)', losses=losses, time=dt, mem_mb=mem, acc=eval_acc(model))

In [ ]:
results = [train_fp32(), train_fp16(), train_bf16()]

print(f'{"配置":<20} {"时间(s)":>10} {"显存峰(MB)":>12} {"准确率":>8}')
print('-' * 56)
for r in results:
    if 'skip' in r:
        print(f'{r["name"]:<20}   <跳过：{r["skip"]}>')
    else:
        print(f'{r["name"]:<20} {r["time"]:>10.2f} {r["mem_mb"]:>12.1f} {r["acc"]*100:>7.2f}%')

In [ ]:
# loss 曲线对比
plt.figure(figsize=(9, 4))
for r in results:
    if r.get('losses'):
        plt.plot(r['losses'], label=r['name'], alpha=0.7)
plt.xlabel('iteration'); plt.ylabel('loss'); plt.title(f'三种精度的 loss 曲线（{EPOCHS} epochs）')
plt.legend(); plt.grid(True); plt.show()

## 3. 学习率 warmup —— 现代训练的另一标配

**问题**：Adam 一开始的二阶动量估计很噪，直接用满学习率，模型早期会被「带飞」（loss 一会涨一会跌甚至 nan）。

**方案**：开始 N 步内线性升到目标 lr，再 cosine 衰减回 0。

In [ ]:
import math

def cosine_with_warmup(step: int, warmup: int, total: int, peak_lr: float, min_lr: float = 0.0):
    if step < warmup:
        return peak_lr * (step + 1) / warmup
    progress = (step - warmup) / max(1, total - warmup)
    return min_lr + 0.5 * (peak_lr - min_lr) * (1 + math.cos(math.pi * progress))

total_steps = 500
warmup = 50
lrs = [cosine_with_warmup(s, warmup, total_steps, 1e-3) for s in range(total_steps)]

plt.figure(figsize=(8, 3))
plt.plot(lrs)
plt.xlabel('step'); plt.ylabel('learning rate')
plt.title('Linear warmup + cosine decay (warmup=50, total=500, peak=1e-3)')
plt.grid(True); plt.show()

In [ ]:
# 真实场景集成：把 warmup 接到 optimizer 上
model, opt = make_model_opt()
scheduler = torch.optim.lr_scheduler.LambdaLR(
    opt, lr_lambda=lambda step: cosine_with_warmup(step, 50, 500, 1.0, 0.0)
    # 注意：LambdaLR 的返回是「乘子」，base_lr 已经在 optimizer 里设过
)

for step in range(0, 500, 100):
    print(f'step {step:3d}  current lr = {scheduler.get_last_lr()[0]:.2e}')
    for _ in range(100):
        scheduler.step()

## 深入思考

1. **什么时候必须用 fp16 而不是 bf16？**
   - 老 GPU（V100 之前）不支持 bf16，只能 fp16。A100 / RTX 30+/40+ 都支持 bf16，**直接用 bf16**。
2. **为什么 fp16 要 `GradScaler` 而 bf16 不要？**
   - fp16 最小正常数 ≈ 6e-5，反传时小梯度直接归 0（underflow）。scaler 把 loss 乘 2^k，让梯度全部进入 fp16 表示范围。bf16 范围与 fp32 同（最小 ≈ 1e-38），不会 underflow。
3. **`autocast` 内部到底干了什么？**
   - 把矩阵乘等「能用低精度」的算子自动切到低精度；softmax/loss 等需要高精度的留 fp32。这是「混合精度」的本意。
4. **grad clip = 1.0 是怎么定的？**
   - 经验值，几乎所有大模型训练都用 0.5~1.0。要点是：**有 vs 无差别很大**，具体数字差别不大。
5. **如果 loss 突然 nan，先查哪 3 件事？**
   - (1) lr 是不是太大 (2) 数据有没有异常值（除 0 / log(0)）(3) fp16 下没用 scaler？

改一改：把 `train_fp16` 里的 `scaler.scale(loss).backward()` 改成 `loss.backward()`（去掉 scaler）跑跑，看 loss 是不是越训越大或 nan。

## 自检 ✅

- [ ] 解释「fp16 vs bf16」的本质差异（指数 vs 尾数）。
- [ ] 解释 GradScaler 的工作原理（scale up → backward → unscale → step）。
- [ ] 解释「warmup」为什么必要。
- [ ] 给一份 loss = nan 的训练日志，能 30 秒内列出 5 个最可能的原因。
- [ ] 解释「为什么 bf16 是现代 LLM 训练的默认」。

## 下一步

→ [`13_attention_deepdive.ipynb`](13_attention_deepdive.ipynb)